In [1]:
import gym
import crafter
import torch.nn as nn
import matplotlib as plt
import torch
import random
from collections import deque
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self, h, w, c, c1 = 32, c2 = 64, c3 = 128, c4 = 256, hidden_size = 1024, kernel_size = 4, stride = 2, padding = 1, dilation = 1, groups = 1):
        super().__init__()
        h1o = (((h + 2 * padding - kernel_size) / stride) + 1)
        h2o = (((h1o + 2 * padding - kernel_size) / stride) + 1)
        h3o = (((h2o + 2 * padding - kernel_size) / stride) + 1)
        h4o = (((h3o + 2 * padding - kernel_size) / stride) + 1)

        w1o = (((w + 2 * padding - kernel_size) / stride) + 1)
        w2o = (((w1o + 2 * padding - kernel_size) / stride) + 1)
        w3o = (((w2o + 2 * padding - kernel_size) / stride) + 1)
        w4o = (((w3o + 2 * padding - kernel_size) / stride) + 1)
        self.net = nn.Sequential(
            nn.Conv2d(c, c1, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups),
            nn.ELU(),
            nn.Conv2d(c1, c2, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups),
            nn.ELU(),
            nn.Conv2d(c2, c3, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups),
            nn.ELU(),
            nn.Conv2d(c3, c4, kernel_size, stride=stride, padding=padding, dilation=dilation, groups=groups),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(c4 * h4o * w4o, hidden_size)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class LatentStateEncoder(nn.Module):
    def __init__(self, h, w, c, num_layers = 2, hidden_size_lstm = 1024, c1 = 32, c2 = 64, c3 = 128, c4 = 256, hidden_size_cnn = 1024, kernel_size = 4, stride = 2, padding = 1, dilation = 1, groups = 1):
        super().__init__()
        self.input_size_lstm = hidden_size_cnn
        self.visual_encoder = CNNEncoder(h, w, c, c1, c2, c3, c4, hidden_size_cnn, kernel_size, stride, padding, dilation)
        self.lstm = nn.LSTM(hidden_size_cnn, hidden_size_lstm, num_layers=num_layers, batch_first=True)
    
    def forward(self, x):
        if len(x.shape) != 5:
            return None
        batch, seq_length, channels, h, w = x.shape
        x = x.reshape(batch * seq_length, channels, h, w)
        encoded_states = self.visual_encoder(x)
        encoded_states = encoded_states.reshape(batch, seq_length, self.input_size_lstm)
        return self.lstm(encoded_states)

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, latent_space_encoder_model, num_layers, hidden_size, num_actions):
        super().__init__()
        self.num_actions = num_actions
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.lse_model = latent_space_encoder_model
        self.actor_net = nn.Sequential(
            nn.Linear(hidden_size, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, num_actions)
        )
        self.critic_net = nn.Sequential(
            nn.Linear(hidden_size, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1)
        )


    def forward(self, x):
        if len(x.shape) != 5:
            return None
        batch_size, seq_length, channel_size, h, w = x.shape
        # lstm_output shape ===> batch_size, seq_length, hidden_size
        lstm_output, h_c_n = self.lse_model(x)
        # h_n shape ===> num_layers, batch_size, hidden_size
        # c_n shape ===> num_layers, batch_size, hidden_size
        h_n, c_n = h_c_n
        last_output = lstm_output[:, -1, :]
        return self.actor_net(last_output), self.critic_net(last_output), lstm_output, h_n, c_n

In [41]:
def take_action(model, state, num_actions):
    logits, value, lstm_out, h_n, c_n = model(state)
    dist = torch.distributions.Categorical(logits=logits[0])
    action = dist.sample()
    return action.item()

In [27]:
env = crafter.Env(seed=42)
env = crafter.Recorder(
    env,
    "../logs",
    save_stats=True,
    save_video=False,
    save_episode=False,
)
env.reset()
obs, reward, done, info = env.step(1)
list_ach = [a for a, _ in info['achievements'].items()]
list_ach

['collect_coal',
 'collect_diamond',
 'collect_drink',
 'collect_iron',
 'collect_sapling',
 'collect_stone',
 'collect_wood',
 'defeat_skeleton',
 'defeat_zombie',
 'eat_cow',
 'eat_plant',
 'make_iron_pickaxe',
 'make_iron_sword',
 'make_stone_pickaxe',
 'make_stone_sword',
 'make_wood_pickaxe',
 'make_wood_sword',
 'place_furnace',
 'place_plant',
 'place_stone',
 'place_table',
 'wake_up']

In [ ]:
obs, reward, done, info

In [29]:
achievement_list = [
    'collect_coal', 'collect_diamond', 'collect_drink', 'collect_iron',
    'collect_sapling', 'collect_stone', 'collect_wood', 'defeat_skeleton',
    'defeat_zombie', 'eat_cow', 'eat_plant', 'make_iron_pickaxe',
    'make_iron_sword', 'make_stone_pickaxe', 'make_stone_sword', 'make_wood_pickaxe',
    'make_wood_sword', 'place_furnace', 'place_plant', 'place_stone',
    'place_table', 'wake_up'
]

In [ ]:
achievement_dict = {ach: index for ach, index in zip(achievement_list, range(len(achievement_list)))}

achievement_inverse_dict = {index: ach for ach, index in achievement_dict.items()}

In [32]:
achievement_dict

{'collect_coal': 0,
 'collect_diamond': 1,
 'collect_drink': 2,
 'collect_iron': 3,
 'collect_sapling': 4,
 'collect_stone': 5,
 'collect_wood': 6,
 'defeat_skeleton': 7,
 'defeat_zombie': 8,
 'eat_cow': 9,
 'eat_plant': 10,
 'make_iron_pickaxe': 11,
 'make_iron_sword': 12,
 'make_stone_pickaxe': 13,
 'make_stone_sword': 14,
 'make_wood_pickaxe': 15,
 'make_wood_sword': 16,
 'place_furnace': 17,
 'place_plant': 18,
 'place_stone': 19,
 'place_table': 20,
 'wake_up': 21}

In [38]:
def play_and_store(
    model, env, seq_length,
    c, h, w, hidden_size,
    achievement_dict,
    device, num_actions, eps=0.3
    ):
    with torch.no_grad():
        er_buffer_obs = torch.empty(0, seq_length, c, h, w).to(device)
        er_buffer_f_obs = torch.empty(0, seq_length, c, h, w).to(device)
        er_buffer_actions = torch.empty(0).to(device)
        er_buffer_reward = torch.empty(0).to(device)
        er_buffer_achievement = torch.empty(0, len(achievement_dict)).to(device)
        er_buffer_latent_space_obs = torch.empty(0, hidden_size).to(device)
        er_buffer_latent_space_f_obs = torch.empty(0, hidden_size).to(device)
        obs = env.reset()
        obs = torch.permute(torch.tensor(obs).to(torch.float32)).to(device)
        tensor_obs = torch.empty(0, c, h, w).to(device)
        for i in range(seq_length - 1):
            obs_zero = torch.zeros(1, c, h, w).to(device)
            tensor_obs = torch.cat([tensor_obs, obs_zero], dim=0)
        tensor_obs = torch.cat([tensor_obs, obs.unsqueeze(0)], dim = 0)
        done = False
        past_info = {k: 0 for k, _ in achievement_dict.items()}
        while done is not True:
            action = take_action(model, tensor_obs[-seq_length :,:,:,:].unsqueeze(0), num_actions, eps)
            obs, reward, done, info = env.step(action)
            obs = torch.permute(torch.tensor(obs).to(torch.float32)).to(device)
            tensor_obs = torch.cat([tensor_obs, obs.unsqueeze(0)], dim = 0)
            er_buffer_obs = torch.cat([er_buffer_obs, obs[-(seq_length - 1) : -1,:,:,:].unsqueeze(0)], dim=0)
            er_buffer_f_obs = torch.cat([er_buffer_f_obs, obs[-seq_length :,:,:,:].unsqueeze(0)], dim=0)
            er_buffer_actions = torch.cat([er_buffer_actions, torch.tensor([action]).to(device)], dim=0)
            er_buffer_reward = torch.cat([er_buffer_reward, torch.tensor([reward]).to(device)], dim = 0)
            achievement_tensor = torch.zeros(len(achievement_dict)).to(device)
            for ach, obtained in info.items():
                if obtained == 1 and past_info[ach] == 0:
                    achievement_tensor[achievement_dict[ach]] = 1.0
            er_buffer_achievement = torch.cat([er_buffer_achievement, achievement_tensor.unsqueeze(0)], dim = 0)
            _, _, lstm_output_obs, _, _ = model(obs[-(seq_length - 1) : -1,:,:,:].unsqueeze(0))
            _, _, lstm_output_f_obs, _, _ = model(obs[-seq_length :,:,:,:].unsqueeze(0))
            er_buffer_latent_space_obs = torch.cat([er_buffer_latent_space_obs, lstm_output_obs], dim = 0)
            er_buffer_latent_space_f_obs = torch.cat([er_buffer_latent_space_f_obs, lstm_output_f_obs], dim = 0)

    return er_buffer_obs, er_buffer_f_obs, er_buffer_actions, er_buffer_reward, er_buffer_achievement, er_buffer_latent_space_obs, er_buffer_latent_space_f_obs
    


In [39]:
def create_experiences(
    model, env, seq_length,
    c, h, w, hidden_size,
    achievement_dict, num_experiences,
    device, num_actions
):
    exp_buffer_obs = []
    exp_buffer_f_obs = []
    exp_buffer_actions = []
    exp_buffer_reward = []
    exp_buffer_achievement = []
    exp_buffer_latent_space_obs = []
    exp_buffer_latent_space_f_obs = []
    for i in range(num_experiences):
        er_buffer_obs, er_buffer_f_obs, er_buffer_actions, er_buffer_reward, er_buffer_achievement, er_buffer_latent_space_obs, er_buffer_latent_space_f_obs = play_and_store(model, env, seq_length, c, h, w, hidden_size, achievement_dict, device, num_actions)
        exp_buffer_obs.append(er_buffer_obs)
        exp_buffer_f_obs.append(er_buffer_f_obs)
        exp_buffer_actions.append(er_buffer_actions)
        exp_buffer_reward.append(er_buffer_reward)
        exp_buffer_achievement.append(er_buffer_achievement)
        exp_buffer_latent_space_obs.append(er_buffer_latent_space_obs)
        exp_buffer_latent_space_f_obs.append(er_buffer_latent_space_f_obs)
    return exp_buffer_obs,
    exp_buffer_f_obs, exp_buffer_actions,
    exp_buffer_reward, exp_buffer_achievement,
    exp_buffer_latent_space_obs, exp_buffer_latent_space_f_obs


In [35]:
# CNN arguments
height = 64
weight = 64
channels_in = 3
channels_hidden_1 = 32
channels_hidden_2 = 64
channels_hidden_3 = 128
channels_hidden_4 = 256
hidden_size_cnn = 1024
kernel_size = 4
stride = 2
padding = 1
dilation = 1
groups = 1

# LSTM arguments
num_layers_lstm = 2
hidden_size_lstm = 1024

# ActorCritic architecture arguments
hidden_size_input = hidden_size_lstm
num_actions = 17

# greediness
eps = 1.0

# discount for greediness
discount_greediness = 0.9978

# after how many steps should update
steps_updating = 2000

# experiences every iteration
num_exp = 32

# total iterations
iterations = 5000

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
if torch.cuda.is_available():
    device = torch.device("cuda")

In [ ]:
lstm_model = LatentStateEncoder(height, weight, channels_in, num_layers_lstm, hidden_size_lstm,
    channels_hidden_1, channels_hidden_2, channels_hidden_3, channels_hidden_4,
    hidden_size_cnn, kernel_size, stride, padding, dilation, groups).to(device)
a2c_model = ActorCritic(lstm_model, num_layers_lstm, hidden_size_lstm).to(device)

In [ ]:
def actor_loss(log_probs, advantages):
    compute_la = log_probs * advantages
    return -(compute_la.mean())

In [ ]:
def critic_loss(advantages):
    return (advantages * advantages).mean()

In [ ]:
def contrastive_loss(lstm_model, ach_episodes,
    contrastive_loss_temperature, num_achievements,
    hidden_size, device):
    numerator = torch.tensor([0])
    denominator = torch.tensor([0])
    tensor_ach = {idx: torch.empty([0, hidden_size]) for idx in range(num_achievements)}
    for key, tensor_observations in ach_episodes.items():
        if tensor_observations.shape[0] == 0:
            continue
        tensor_latent_observations = lstm_model(tensor_observations)
        tensor_ach[key] = torch.cat([tensor_ach[key], tensor_latent_observations], dim=0)
    for key, tensor_latent in tensor_ach.items():
        n = tensor_latent.shape[0]
        z = F.normalize(tensor_latent, p=2, dim=1) # n, hidden
        similarities = z @ z.T
        sim_w_temp = similarities / contrastive_loss_temperature
        scores = sim_w_temp.exp()
        sim_w_temp[[range(n)], range(n)] = 10000.0
        sim_w_temp = torch.where(sim_w_temp < 10000.0, sim_w_temp, 0.0)
        numerator = numerator + sim_w_temp.triu(diagonal=1).sum()
    
    # compute denominator
    # convert to tuple list (key, tensor_latent)
    # for each -> take tensor latent
    # for each tensor latent
    # go to the remaining tuple list
    # create torch with the selected tensor as first
    # compute cosine sim
    # select only first row
    ach_list_tuples = [(key, tensor_latent) for key, tensor_latent in tensor_ach.items() if tensor_latent.shape[0] > 0]
    length_list = len(ach_list_tuples)
    # OR MAKE A GIANT MATRIX AND RETAIN INDICES NEEDED
    ach_indices = []
    all_tensors_combined = torch.empty(0, hidden_size).to(device)
    for key, tensor_latent in ach_list_tuples:
        old_size = all_tensors_combined.shape[0]
        all_tensors_combined = torch.cat([all_tensors_combined, tensor_latent], dim=0)
        new_size = all_tensors_combined.shape[0]
        ach_indices.append([key, old_size, new_size])
    z = F.normalize(all_tensors_combined, p=2, dim=1)
    sims = z @ z.T
    sims = sims / contrastive_loss_temperature
    sims = sims.exp()
    sims[range(sims.shape[0]), range(sims.shape[0])] = 0.0
    sims = sims.triu(diagonal=1)
    for key, start, stop in ach_indices:
        # aduna la numarator si numitor pentru fractia de loss
        sims_key_ach = sims[start:stop, start:stop]
        numerator = numerator + sims_key_ach.sum()
        sims_distinct_key_ach = sims[start:stop, stop:]
        denominator = denominator + sims_distinct_key_ach.sum()
        pass




In [ ]:
def train_model(
    a2c_model, lstm_model, env,
    steps_to_play, games_per_play,
    optim_lr, centered, alpha, eps_optim,
    seq_length, c, h, w, hidden_size,
    achievement_dict, device, num_actions,
    game_batch_size, discount, actor_loss_weight,
    critic_loss_weight, entropy_loss_weight,
    prob_contrastive, warm_up_steps, contrastive_loss_batch_size,
    contrastive_loss_temperature
):
    optimizer = torch.optim.RMSprop(
        a2c_model.parameters(), lr=optim_lr,
        centered=centered, alpha=alpha, eps=eps
    )

    achievement_tensors_list = []
    for i in range(len(achievement_dict)):
        ach_tensor = torch.zeros(len(achievement_dict))
        ach_tensor[i] = 1.0

    for play_step in range(steps_to_play):
        buffer_list_tuple = create_experiences(a2c_model, env, seq_length, c, h, w, hidden_size,
            achievement_dict, games_per_play, device, num_actions)
        if play_step >= warm_up_steps:
            if random.choices(population=[0, 1], weights=[1 - prob_contrastive, prob_contrastive])[0] == 1:
                # what needs here:
                # 1. get how many in list
                # 2. shuffle indices of list
                # 3. get in batches
                # 4. get tensors from those batches
                # 5. do the loss and gradient descent
                len_steps = buffer_list_tuple[0][game].shape[0]
                indices = torch.randperm(len_steps)
                tensor_obs = buffer_list_tuple[0][game][indices]
                tensor_f_obs = buffer_list_tuple[1][game][indices]
                tensor_act = buffer_list_tuple[2][game][indices]
                tensor_reward = buffer_list_tuple[3][game][indices]
                tensor_ach = buffer_list_tuple[4][game][indices]
                tensor_latent = buffer_list_tuple[5][game][indices]
                tensor_f_latent = buffer_list_tuple[6][game][indices]
                for i in range(0, len_steps, contrastive_loss_batch_size):
                    ach_episodes = {idx: torch.empty([0, seq_length, c, h, w]) for idx in range(len(achievement_dict))}
                    for i in range(len(achievement_dict)):
                        idx = torch.where(tensor_ach[:, i] == 1.0)[0]
                        selected_tensor_obs = tensor_obs[idx]
                        ach_episodes[i] = torch.cat([ach_episodes[i], selected_tensor_obs], dim = 0)
                    
                # # make 22 tensors to add for every achievement
                # ach_episodes = {idx: torch.empty([0, seq_length, c, h, w]) for idx in range(len(achievement_dict))}
                # # one episode: formed by achievement and states
                # for game in range(games_per_play):
                #     # search to see in a game which seq of states lead to a reward
                #     # shape ===> total_iterations, seq_length, 3, 64, 64
                #     tensor_obs = buffer_list_tuple[0][game]
                #     # shape ===> total_iterations, 1
                #     tensor_reward = buffer_list_tuple[3][game]
                #     # shape ===> total_iterations, 22
                #     tensor_ach = buffer_list_tuple[4][game]
                #     for i in range(len(achievement_dict)):
                #         idx = torch.where(tensor_ach[:, i] == 1.0)[0]
                #         selected_tensor_obs = tensor_obs[idx]
                #         ach_episodes[i] = torch.cat([ach_episodes[i], selected_tensor_obs], dim = 0)
                
                    
        for game in range(games_per_play):
            len_steps = buffer_list_tuple[0][game].shape[0]
            indices = torch.randperm(len_steps)
            tensor_obs = buffer_list_tuple[0][game][indices]
            tensor_f_obs = buffer_list_tuple[1][game][indices]
            tensor_act = buffer_list_tuple[2][game][indices]
            tensor_reward = buffer_list_tuple[3][game][indices]
            tensor_ach = buffer_list_tuple[4][game][indices]
            tensor_latent = buffer_list_tuple[5][game][indices]
            tensor_f_latent = buffer_list_tuple[6][game][indices]
            for i in range(0, len_steps, game_batch_size):
                optimizer.zero_grad()
                actor_output, critic_output, lstm_output, h_n, c_n = a2c_model(tensor_obs[i: min(game_batch_size + i, len_steps)])
                _, critic_output_f_state, _, _, _ = a2c_model(tensor_f_obs[i: min(game_batch_size + i, len_steps)])
                advantages = (tensor_reward[i: min(game_batch_size + i, len_steps)] + discount * critic_output_f_state) - critic_output
                distributions = torch.distributions.Categorical(logits=actor_output)
                actions = distributions.sample()
                log_probs = distributions.log_probs(actions)
                loss = actor_loss(log_probs, advantages) + critic_loss(advantages) + distributions.entropy().mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(a2c_model.parameters(), 0.5)
                optimizer.step()

    pass

In [12]:
model = LatentStateEncoder(64, 64, 3)
a2c = ActorCritic(model, 2, )
tensor_int = torch.randint(low=0, high=256, size=(1, 3, 3, 64, 64)).to(torch.float32)
h_n, c_n = model(tensor_int)
print(h_n.shape)
print(c_n[0].shape)
print(c_n[1].shape)

torch.Size([1, 3, 1024])
torch.Size([4, 1, 1024])
torch.Size([4, 1, 1024])


In [5]:
model = CNNEncoder(64, 64, 3)
tensor_int = torch.randint(low=0, high=256, size=(1, 3, 64, 64)).to(torch.float32)
res = model(tensor_int)
res.shape

torch.Size([1, 1024])